# Calibration viewer: raw vs calibrated probabilities

Reads the small per-case files written by `build_display_cache.py`. It never runs the model and never opens the
large logit files. Run that script once per cache first:

```
python Analyses/Calibration/build_display_cache.py --cache data/cache/851c43ca5339
python Analyses/Calibration/build_display_cache.py --cache data/cache_confirm/851c43ca5339
```

**What is exact and what is approximate**
- The images are downsampled (every 2nd voxel per axis) and the probabilities are stored as 8-bit values. They are for display only.
- Precision, recall and Dice at the slider threshold come from full-resolution histograms of each case
  (100 threshold steps, the same rule as `evaluate_calibration.py`). They are never computed from the downsampled images.
- Every figure is titled with its key: case, checkpoint hash, calibrator hash, probability set and class.

**Raw vs calibrated.** "Calibrated" is `calib_pred8_predclass.json`: one temperature per predicted class, applied
within 8 voxels of the predicted organ. It cannot change the argmax segmentation, so argmax Dice is the same for both.
It does change *which voxels pass a threshold t*, so the mask at a fixed t, and the sweep curves, do move.

**Confirmation cases.** The 316 confirmation patients were evaluated once, as pre-registered in
`CONFIRMATION_PLAN.md`. Anything you find by browsing them here is exploratory, not part of the confirmed result.

In [ ]:
import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.size"] = 11

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
CACHES = {
    "exploratory test (49)": REPO / "data/cache/851c43ca5339",
    "confirmation (316)": REPO / "data/cache_confirm/851c43ca5339",
}
CACHE_NAME = "confirmation (316)"   # change to "exploratory test (49)" to browse the other set

DISP = CACHES[CACHE_NAME] / "display"
if not (DISP / "index.json").exists():
    ready = [n for n, c in CACHES.items() if (c / "display/index.json").exists()]
    raise FileNotFoundError(
        f"No display cache for '{CACHE_NAME}' yet ({DISP}/index.json is written when build_display_cache.py finishes).\n"
        f"Ready now: {ready}. Set CACHE_NAME to one of those, or run:\n"
        f"  python Analyses/Calibration/build_display_cache.py --cache {CACHES[CACHE_NAME].relative_to(REPO)}")
INDEX = json.load(open(DISP / "index.json"))
CAL_NAME = INDEX["calibrator_name_in_metrics"]          # key of the calibrator inside metrics.json per_case
CLASSES = INDEX["classes"]                               # kidney, tumor, cyst  (labels 1, 2, 3)
COLORS = {"kidney": "#e0703f", "tumor": "#3f7fd6", "cyst": "#2b7a4b"}
CASES = [r["case"] for r in INDEX["cases"]]
ROWS = {r["case"]: r for r in INDEX["cases"]}
print(f"{CACHE_NAME}: {len(CASES)} cases · checkpoint {INDEX['checkpoint_sha256'][:12]} · "
      f"calibrator {INDEX['calibrator']} ({INDEX['calibrator_sha256'][:8]}) · display every {INDEX['down']}th voxel")

## 1 · Per-case results (pooling hides cases)

The confirmation numbers are pooled over all voxels of all patients, so large kidneys dominate them.
These panels show one point per patient instead.
- **(a) and (b):** organ-region NLL and top-label ECE, raw against calibrated. Points below the diagonal improved.
- **(c):** per-case tumor Dice against tumor volume. Small tumors are where the model fails, and pooled Dice hides that.

In [ ]:
def per_case(metric, prob_set):
    out = {}
    for c in CASES:
        m = ROWS[c].get("metrics", {}).get(prob_set, {})
        v = m.get(metric) if not metric.startswith("dice_") else (m.get("argmax_dice") or {}).get(metric[5:])
        if v is not None:
            out[c] = v
    return out

@lru_cache(maxsize=None)
def n_label_voxels(case):
    with np.load(DISP / f"{case}.npz") as z:   # npz members load lazily: only the small histogram is read
        return z["hist_raw"][:, 1].sum(axis=1)  # voxels of kidney, tumor, cyst at full resolution

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, metric, name in [(axes[0], "nll_fg", "organ-region NLL"), (axes[1], "ece_top_fg", "organ-region top-label ECE")]:
    raw, cal = per_case(metric, "T1"), per_case(metric, CAL_NAME)
    cs = sorted(set(raw) & set(cal))
    x, y = np.array([raw[c] for c in cs]), np.array([cal[c] for c in cs])
    lim = [min(x.min(), y.min()) * 0.9, max(x.max(), y.max()) * 1.1]
    ax.plot(lim, lim, ls="--", c="0.5", lw=1)
    ax.scatter(x, y, s=12, alpha=0.6, c="#0b6e8a")
    ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="raw", ylabel="calibrated",
           title=f"{name}\n{(y < x).sum()}/{len(cs)} patients improved · median change {np.median(y - x):+.4f}")
    for a in (ax.xaxis, ax.yaxis):
        a.set_minor_formatter(mpl.ticker.NullFormatter())

dice = per_case("dice_tumor", "T1")
cs = sorted(dice)
vol_ml = np.array([n_label_voxels(c)[1] for c in cs]) * 1.5 ** 3 / 1000
d = np.array([dice[c] for c in cs])
ok = vol_ml > 0
axes[2].scatter(vol_ml[ok], d[ok], s=12, alpha=0.6, c=COLORS["tumor"])
axes[2].set(xscale="log", xlabel="tumor volume (ml, resampled label)", ylabel="argmax tumor Dice (per case)",
            title=f"tumor Dice per case\nmean {d[ok].mean():.3f} · median {np.median(d[ok]):.3f} · "
                  f"{(d[ok] == 0).sum()} cases with Dice 0")
fig.suptitle(f"{CACHE_NAME} · raw vs {CAL_NAME}", y=1.02)
plt.tight_layout()
plt.show()

## 2 · Slice viewer with threshold slider

- **Overlay: probability.** p(class) as a heat map. Values below 0.02 are hidden.
- **Overlay: mask.** Voxels with p(class) ≥ t.
- **Overlay: TP / FP / FN.** The same mask compared with the ground truth.
- **GT outline.** A white contour of the labelled class.
- **Right-hand panel.** This case's full-resolution precision, recall and Dice against t, raw (dashed) and calibrated (solid).
  The numbers in the title are read from it at the current t.

Changing the case moves the slice sliders to the centre of that case's labelled class, or to the volume centre if the class is absent.

In [ ]:
@lru_cache(maxsize=6)
def load_case(case):
    z = np.load(DISP / f"{case}.npz")
    return {k: z[k] for k in ("ct", "label", "p_raw", "p_cal", "hist_raw", "hist_cal")} | {"key": json.loads(str(z["key"]))}

def sweep(H):
    # H: (2, B+1) counts by bin; row 1 = voxels of this class. Returns P, R, Dice of p >= j/B for j = 0..B.
    tp = H[1, ::-1].cumsum()[::-1].astype(float)
    fp = H[0, ::-1].cumsum()[::-1].astype(float)
    fn = H[1].sum() - tp
    with np.errstate(invalid="ignore", divide="ignore"):
        return tp / (tp + fp), tp / (tp + fn), 2 * tp / (2 * tp + fp + fn)

TP_C, FP_C, FN_C = np.array([26, 154, 133]) / 255, np.array([210, 51, 111]) / 255, np.array([226, 160, 0]) / 255

def overlay(p, lab, k, t, mode):
    rgba = np.zeros(p.shape + (4,))
    if mode == "probability":
        rgba[:] = mpl.colormaps["magma"](p)
        rgba[..., 3] = np.where(p > 0.02, 0.25 + 0.6 * p, 0)
        return rgba
    m = p >= t
    if mode == "mask":
        rgba[m] = (*mpl.colors.to_rgb(COLORS[CLASSES[k]]), 0.65)
        return rgba
    g = lab == k + 1
    rgba[m & g] = (*TP_C, 0.6); rgba[m & ~g] = (*FP_C, 0.85); rgba[~m & g] = (*FN_C, 0.85)
    return rgba

def draw(case, cls, prob_set, t, mode, gt, ax_i, co_i, sa_i):
    D = load_case(case); key = D["key"]
    k = CLASSES.index(cls)
    p = D["p_cal" if prob_set == "calibrated" else "p_raw"][k] / 255.0
    lab, ct = D["label"], D["ct"]
    B = key["bins"]; j = int(round(t * B))

    fig = plt.figure(figsize=(17, 5.2))
    gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.15])
    views = [("axial", lambda a: a[:, :, ax_i]), ("coronal", lambda a: a[:, co_i, :]), ("sagittal", lambda a: a[sa_i, :, :])]
    for i, (name, cut) in enumerate(views):
        ax = fig.add_subplot(gs[i])
        ax.imshow(cut(ct).T, cmap="gray", origin="lower", vmin=0, vmax=255)
        ax.imshow(overlay(cut(p), cut(lab), k, t, mode).transpose(1, 0, 2), origin="lower")
        if gt and (cut(lab) == k + 1).any():
            ax.contour((cut(lab) == k + 1).T.astype(float), levels=[0.5], colors="white", linewidths=0.9)
        ax.set_title(f"{name} {[ax_i, co_i, sa_i][i]}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

    ax = fig.add_subplot(gs[3])
    tt = np.arange(B + 1) / B
    out = {}
    for s, ls in (("raw", "--"), ("calibrated", "-")):
        P, R, Dc = sweep(D["hist_cal" if s == "calibrated" else "hist_raw"][k])
        out[s] = (P[j], R[j], Dc[j])
        for y, c, nm in ((P, "#5446b8", "precision"), (R, "#e2a000", "recall"), (Dc, "#0b6e8a", "Dice")):
            ax.plot(tt, y, ls=ls, c=c, lw=1.6, label=f"{nm} ({s})")
    ax.axvline(t, c="k", lw=1, ls=":")
    ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel=f"threshold t on p({cls})", title="full-resolution sweep, this case")
    ax.legend(fontsize=7.5, ncol=2, loc="lower center")

    m = ROWS[case].get("metrics", {})
    ad = (m.get("T1", {}).get("argmax_dice") or {}).get(cls)
    Pj, Rj, Dj = out[prob_set]
    fmt = lambda v: "—" if v is None or not np.isfinite(v) else f"{v:.3f}"
    n_cls = int(D["hist_raw"][k, 1].sum())
    fig.suptitle(
        f"{case} · ckpt {key['checkpoint_sha256'][:12]} · calibrator {key['calibrator_sha256'][:8]} · {prob_set} · {cls}"
        f" · display 1/{key['down']} per axis\n"
        f"at t = {j / B:.2f}: precision {fmt(Pj)}  recall {fmt(Rj)}  Dice {fmt(Dj)}   "
        f"(raw {fmt(out['raw'][2])} / calibrated {fmt(out['calibrated'][2])})   ·   argmax Dice {fmt(ad)}"
        f"   ·   {n_cls:,} {cls} voxels" + ("" if n_cls else " (class absent: precision only counts false positives)"),
        fontsize=10.5, y=1.03)
    plt.tight_layout()
    plt.show()

In [ ]:
st = dict(continuous_update=False, layout=w.Layout(width="420px"))
case_w = w.Dropdown(options=CASES, description="case")
cls_w = w.ToggleButtons(options=CLASSES, value="tumor", description="class")
set_w = w.ToggleButtons(options=["raw", "calibrated"], value="calibrated", description="probs")
mode_w = w.ToggleButtons(options=["probability", "mask", "TP / FP / FN"], value="TP / FP / FN", description="overlay")
t_w = w.FloatSlider(value=0.5, min=0.01, max=1.0, step=0.01, description="threshold t", **st)
gt_w = w.Checkbox(value=True, description="GT outline")
ax_w = w.IntSlider(description="axial", **st)
co_w = w.IntSlider(description="coronal", **st)
sa_w = w.IntSlider(description="sagittal", **st)

def recentre(*_):
    shape = load_case(case_w.value)["ct"].shape
    lab = load_case(case_w.value)["label"] == CLASSES.index(cls_w.value) + 1
    c = np.argwhere(lab).mean(axis=0).round().astype(int) if lab.any() else np.array(shape) // 2
    for s, n, v in ((sa_w, shape[0], c[0]), (co_w, shape[1], c[1]), (ax_w, shape[2], c[2])):
        s.max = n - 1
        s.value = int(v)

case_w.observe(recentre, names="value")
cls_w.observe(recentre, names="value")
recentre()

ui = w.VBox([w.HBox([case_w, gt_w]), cls_w, set_w, mode_w, t_w, w.HBox([ax_w, co_w, sa_w])])
out = w.interactive_output(draw, dict(case=case_w, cls=cls_w, prob_set=set_w, t=t_w, mode=mode_w, gt=gt_w,
                                      ax_i=ax_w, co_i=co_w, sa_i=sa_w))
display(ui, out)